## Wiederholung & Vertiefung Backpropagation

### 1. Setup & Forward Pass

Wir bauen ein sehr kleines "Netzwerk".
Der DAG entsteht erst beim Ausführen!

In [1]:
import torch

# 1. Input & Parameter
# requires_grad=True -> "Beobachte mich!"
x = torch.tensor(3.0)                       # Input (Konstante)
w = torch.tensor(2.0, requires_grad=True)   # Gewicht
b = torch.tensor(1.0, requires_grad=True)   # Bias

# 2. Forward Computation
# PyTorch baut hier live den DAG (Directed Acyclic Graph)
z = w * x     # z = 6
y = z + b     # y = 7
L = y ** 2    # Loss = 49

print(f"Loss: {L.item()}")

Loss: 49.0


### 2. Introspektion des Graphen

Was hat PyTorch im Hintergrund gebaut?
Wir schauen uns die `grad_fn` (Gradient Function) an.

In [2]:
# Der Kopf des Graphen
print(f"Node L: {L.grad_fn}")  # <PowBackward0>

# Wer kommt als nächstes? (Topological Sort ist implizit)
print(f"Next Node (von L): {L.grad_fn.next_functions[0][0]}") # <AddBackward0>

# Und davor?
print(f"Next Node (von y): {L.grad_fn.next_functions[0][0].next_functions}") 
# Zeigt auf <MulBackward0> (z) und <AccumulateGrad> (b)

Node L: <PowBackward0 object at 0x7fae5838e530>
Next Node (von L): <AddBackward0 object at 0x7fae5838ea10>
Next Node (von y): ((<MulBackward0 object at 0x7fae5838e500>, 0), (<AccumulateGrad object at 0x7fae5838e980>, 0))


Die Variable `grad_fn` weiß, wie man ableitet, und hält Referenzen auf die *saved tensors*
(die gespeicherten Zwischenergebnisse für die Ableitung).

### 3. Backward Pass

Der rekursive Aufruf der Kettenregel.

Initial: $dL/dL = 1$

Regel:
$$dL/dw = dL/dy * dy/dz * dz/dw$$

Hier also:
$$ dL/dw = 2y * 1 * x = 14 * 3 = 42 $$

In [3]:
L.backward()
print(f"dL/dw (w.grad): {w.grad}")
print(f"dL/db (b.grad): {b.grad}")

dL/dw (w.grad): 42.0
dL/db (b.grad): 14.0


### 4. Optimizer Step (SGD)

Wo `torch.no_grad()` notwendig ist.

In [4]:
lr = 0.01

# WICHTIG: Kontext-Manager, damit der Update-Schritt nicht Teil des Graphen wird!
with torch.no_grad():
    # Gradient Descent Update
    w -= lr * w.grad
    b -= lr * b.grad
    
    # Reset der Gradienten für den nächsten Loop (sonst wird akkumuliert!)
    w.grad.zero_()
    b.grad.zero_()

print(f"Neues w: {w:.4f}")
print(f"Neues b: {b:.4f}")

Neues w: 1.5800
Neues b: 0.8600


### 5. Sonderfall: Freezing

Was passiert, wenn wir den Gradienten deaktivieren?

In [5]:
# Einfrieren eines Parameters
w.requires_grad = False

z = w * x

print(f"z grad_fn: {z.grad_fn}")
print(f"y grad_fn: {y.grad_fn}")

z grad_fn: None
y grad_fn: <AddBackward0 object at 0x7fae5838c6a0>


Ergebnis: `z` hat keine grad_fn mehr (Leaf Node im Graphen ist abgeschnitten).

Wenn wir jetzt SGD (oder einen anderen Optimierer) laufen lassen, wird `w` nicht mehr verändert, `b` aber schon.

### 6. Convolution Backprop

Wie funktioniert Backprop,
wenn Gewichte geteilt werden (Shared Weights) wie bei einem Convolution Layer?

In [6]:
# 1D Beispiel
# Input: [1, 2, 3]
input_seq = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)

# Kernel (Gewicht): [0.5] -> wird über den Input geschoben
kernel = torch.tensor([0.5], requires_grad=True)

# Manuelle Convolution (wie nn.Conv1d)
o1 = input_seq[0] * kernel  # 1.0 * 0.5
o2 = input_seq[1] * kernel  # 2.0 * 0.5
o3 = input_seq[2] * kernel  # 3.0 * 0.5

loss = o1 + o2 + o3
loss.backward()

# Der Gradient des Kernels ist die Summe der Gradienten an jeder Position
# dL/dkernel = x1 + x2 + x3 = 1 + 2 + 3 = 6
print(f"Kernel Gradient: {kernel.grad}")

Kernel Gradient: tensor([6.])
